# Agentic Workshop: Building Cheminformatics Agents with LangGraph

**Last updated:** 2026 • *Built on the original workshop materials by Deepa Korani.*
Author for QSAR Segment: Ricardo Tieghi

## What you'll build

By the end of this notebook you will have built **two AI agents** using **LangGraph** and **Google Gemini**:

1. A simple **math agent** — to learn how tool-calling works.
2. A **QSAR-modeling agent** — that loads a real skin-sensitization dataset, builds Random Forest and SVM classifiers, evaluates them, and predicts whether a new molecule is a sensitizer.

## Roadmap

| Part | Topic |
| --- | --- |
| 1 | Setup & imports |
| 2 | What is an *agent*? |
| 3 | A simple tool-calling agent (math) |
| 4 | A QSAR-modeling agent (skin sensitization) |

> **Heads-up:** Parts 1–3 introduce the building blocks. Part 4 is where the cheminformatics happens — we connect a real ML pipeline to an agent that you can talk to in plain English.

# PART 1: Setup and Imports

We need three things before we can build agents:

1. **A Gemini API key** — the agent's brain.
2. **Python libraries** — LangGraph (the agent framework), RDKit (chemistry), scikit-learn / Optuna (machine learning).
3. **Standard imports** — we pull in everything once so the rest of the notebook stays clean.

> 🔑 **Get a free Gemini API key** at https://aistudio.google.com/apikey. Paste it into the prompt that pops up below — it is stored only in this Python session, not in the notebook file.

> ⚠️ **Never share your API key** or commit it to a git repo.

In [1]:
import os
from getpass import getpass

# One key, used by every Gemini-powered cell in this notebook.
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

print("✅ API key loaded.")

Enter your Gemini API key: ··········
✅ API key loaded.


### Install the libraries we'll use

Run this **once** per environment. If you've already installed everything, the cell will be quick.

- `langchain`, `langgraph`, `langchain-google-genai` — agent framework + Gemini binding
- `rdkit` — cheminformatics (SMILES → fingerprints)
- `scikit-learn`, `optuna`, `imbalanced-learn`, `joblib` — the QSAR ML stack
- `pandas`, `numpy`, `matplotlib` — data handling and plotting

In [2]:
%pip install -q \
    langchain langchain-google-genai langgraph \
    google-generativeai \
    rdkit \
    scikit-learn optuna imbalanced-learn joblib \
    pandas numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 14.0 MB/s eta 0:00:00


### Quick sanity-check the Gemini connection

If this prints a friendly response, the API key works.

In [3]:
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
test_model = genai.GenerativeModel("models/gemini-2.5-flash")
print(test_model.generate_content("Say hello from Gemini in one short sentence.").text)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Hello from Gemini!


### Main imports

Everything the rest of the notebook needs, in one place. Read the comments to understand which library each piece comes from.

In [4]:
# --- Standard library --------------------------------------------------
from typing import TypedDict, Annotated, List, Dict
import json

# --- LangGraph: the agent runtime --------------------------------------
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import tools_condition, ToolNode
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# --- LangChain: LLM wrapper + message types ----------------------------
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import (
    AnyMessage, SystemMessage, HumanMessage, AIMessage, ToolMessage,
)
from langchain_core.tools import tool

# --- Cheminformatics ---------------------------------------------------
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, PandasTools

# --- Data + ML stack (used in Part 4) ----------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score,
    confusion_matrix, recall_score,
)
from imblearn.metrics import geometric_mean_score

# Optuna prints a lot during hyperparameter search; quiet it for the demo.
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("✅ All imports successful.")

✅ All imports successful.


# PART 2: What is an Agent?

## Agent = LLM + Tools + Reasoning Loop

An **agent** is more than just an LLM answering a question. It's a system that:

1. Reads a user request.
2. Decides whether it has the answer or whether it should *call a tool*.
3. Calls tools (Python functions you wrote), reads the results.
4. Loops back to step 2 until it has enough information to answer.

```
  user question
       ↓
  +-----------+      no tool needed
  |    LLM    | -----------------------> answer
  +-----------+
       ↓ needs a tool
  +-----------+
  |   Tool    | <— your Python function (RDKit, ML model, web API…)
  +-----------+
       ↓
  results sent back to LLM — loop continues
```

## Why LangGraph?

We use **LangGraph** because it makes the loop above *explicit and inspectable*: every tool call is a node in a graph, and you can see exactly which path the agent took.

## Agentic systems in cheminformatics — selected literature

- **ChemCrow** — [Bran et al., 2024](https://doi.org/10.1038/s42256-024-00832-8). An autonomous chemistry agent built on GPT-4 that interacts with cheminformatics tools and databases.
- **CACTUS: Chemistry Agent Connecting Tool-Usage to Science** — [McNaughton et al., 2024](https://pubs.acs.org/doi/10.1021/acsomega.4c08408). An LLM agent integrating RDKit and other tools for similarity search and property prediction.
- **ChemToolAgent: The Impact of Tools on Language Agents for Chemistry Problem Solving** — [Yu et al., 2025](https://arxiv.org/html/2411.07228v3). Evaluates when and how external tools improve LLM performance on chemistry tasks.
- **ChemAgent: Tree-Search-Based Tool Learning for Chemistry and Materials Science** — [Wu et al., 2025](https://arxiv.org/abs/2506.07551). An agentic framework integrating 137 external tools with hierarchical Monte Carlo tree search.

# PART 3: A Simple Tool-Calling Agent

Before we touch chemistry, let's build the smallest possible agent: one that does math by calling Python functions.

## The `@tool` decorator

`@tool` (from LangChain) turns a regular Python function into something an LLM can call. It does four things automatically:

1. Reads the function name, parameters, and **docstring**.
2. Builds a JSON schema the LLM can understand.
3. Handles serialization of inputs and outputs.
4. Makes the function callable by the agent.

**Two requirements you must respect:**
- Every parameter needs a **type hint** (e.g. `a: float`).
- The **docstring** must clearly explain what the function does — the LLM uses it to decide *when* to call this tool.

### Define two tools

We'll define `add` and `subtract`. Look at `subtract` carefully — there is something deliberately wrong with it. We'll come back to that.

In [20]:
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b


@tool
def subtract(a: float, b: float) -> float:
    """Subtract the second number from the first number (a - b)."""
    return a - b

@tool
def multiply(a: float, b: float) -> float:
  """Multiply the two numbers together (a*b)"""
  return a * b

### Defining agent **state**

Every agent needs to remember what's been said. LangGraph models that as a **state** — a dictionary the agent reads and writes as it runs.

Our state has just one field: `messages`.

In [21]:
class SimpleAgentState(TypedDict):
    messages: Annotated[List[AnyMessage], add_messages]

**Breaking that down:**

- `TypedDict` — a Python type hint that defines the structure.
- `messages` — the key that holds our conversation.
- `List[AnyMessage]` — a list of `HumanMessage`, `AIMessage`, `ToolMessage`, or `SystemMessage`.
- `Annotated[..., add_messages]` — the important bit. `add_messages` is a **reducer**: it tells LangGraph to *append* new messages instead of *replacing* the list.

```
Without add_messages:    state["messages"] = new_message    # ❌ overwrites!
With add_messages:       state["messages"].append(new_message) # ✅ keeps history
```

**Visualization:**
```
Initial:   messages = []
After Q1:  messages = [HumanMessage("Hi"), AIMessage("Hello")]
After Q2:  messages = [HumanMessage("Hi"), AIMessage("Hello"),
                       HumanMessage("What is 2+2?"), AIMessage("4")]
```

### Build the agent graph

Two nodes:

- **`agent`** — calls the LLM. The LLM decides whether to answer directly or to call a tool.
- **`tools`** — actually runs whichever tool the LLM picked.

Edges:

- `agent` → `tools` (only if the LLM asked for a tool)
- `tools` → `agent` (always loop back so the LLM can use the result)

In [25]:
def create_simple_agent():
    # Initialize Gemini. temperature=0 makes responses deterministic for demos.
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        google_api_key=os.environ.get("GOOGLE_API_KEY"),
        convert_system_message_to_human=True,  # Gemini quirk
    )
    tools = [add, subtract, multiply]
    llm_with_tools = llm.bind_tools(tools)

    # The agent node: send conversation to the LLM, get a response.
    def call_model(state: SimpleAgentState):
        response = llm_with_tools.invoke(state["messages"])
        return {"messages": [response]}

    # Build the graph.
    graph = StateGraph(SimpleAgentState)
    graph.add_node("agent", call_model)
    graph.add_node("tools", ToolNode(tools))
    graph.add_conditional_edges("agent", tools_condition)  # agent → tools or END
    graph.add_edge("tools", "agent")                      # tools → agent
    graph.set_entry_point("agent")

    return graph.compile()

### Test 1: ask the agent to subtract

We'll ask `"What is 10 minus 3?"`. The right answer is `7`.

**What do you predict the agent will say?**

In [26]:
simple_agent = create_simple_agent()
result = simple_agent.invoke({
    "messages": [HumanMessage(content="What is 10 minus 3?")]
})

print("Final answer:", result["messages"][-1].content)
print("Expected:     7")

Final answer: 10 minus 3 is 7.
Expected:     7


### Test 2: a multi-step query

Watch how the agent chains *two* tool calls.

In [27]:
result = simple_agent.invoke({
    "messages": [HumanMessage(content="What is 23 times 27, plus 5?")]
})

for m in result["messages"]:
    print(f"[{type(m).__name__}] {m.content if m.content else m}")

print("\nFinal answer:", result["messages"][-1].content)

[HumanMessage] What is 23 times 27, plus 5?
[AIMessage] content='' additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 23, "b": 27}'}, '__gemini_function_call_thought_signatures__': {'872427a2-98d0-47c6-a438-95f85712a18d': 'CuQCAQw51sckJOJqcys2L0rUUIZFSnAf/k8ldNaY/Qp62aRoBxvJf6wANOxgqr7o5ZyaXIuroeLLB7ko7UmGx8LCYCToiqwC23Nv5F1Gc1DcheFM9j4nc6sYJgeA7Fj2wRnnYOQNVvOCtd9w3A6AzuTJg08YbMYyEjEfw5E5TYLct0mFZ0lXW7RE2uW24/WMe4nsOwuut4PsCz0DpZpceXsJHkv7BgjcisptaJGkWG3tajNfHY8mPv12QmK7wjYJbZWhZfbsA1Z2CgnDy3IO6XR3PByg+psZdazh8ZQtRmNUsmq+T//xQ7Xth0b1IkhMOSCvVnnvumjAxKhAp21+UpCCduOvHlJLMjs89Ow2bFXHW0kEkCGa6P6qk9JETcP/5XdPflhq1BGvYNnaYF22nGW4ukNxF9xzCLUkLbWsfoQsKS+j8H1lj/V1O0fbYWxck7xs5QJiSxzxE1ODZyBh1rSuTuaDMqY='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019de911-030c-7401-aeb7-f45cb790a6a0-0' tool_calls=[{'name': 'multiply', 'args': {'a': 23, 'b': 27}, 'id': '872427a2-98d0

Notice the trace: the agent decided which tools to call, in what order, and combined their results into one final answer. **That's the agent loop in action.**

# PART 4: A QSAR-Modeling Agent

Now we'll build a real cheminformatics agent. Instead of math tools, this agent will have access to a **machine-learning pipeline** for QSAR modeling.

## What is QSAR?

**QSAR** = *Quantitative Structure–Activity Relationship*. The idea: predict a property of a molecule (toxicity, activity, solubility…) directly from its chemical structure. We turn each molecule into a **numerical fingerprint** and train a classifier to map fingerprint → property.

## Our endpoint: skin sensitization

**Skin sensitization** is an allergic response — some chemicals (e.g. nickel, certain dyes, formaldehyde) cause your immune system to react on repeat exposure. Identifying sensitizers is critical in cosmetics, pharma, and consumer-product safety.

## The dataset

We use the **STopTox skin-sensitization dataset** (`Skin_sensitization.sdf`):

- ~7,271 molecules
- Columns: `CASRN`, `Compound name`, `Smiles`, `InChI Key`, `Outcome`
- `Outcome` is binary: **1** = sensitizer, **0** = non-sensitizer

## Plan

1. **4.1** Define plain Python helper functions — the QSAR pipeline.
2. **4.2** Run the pipeline manually so we know it works.
3. **4.3** Wrap each helper as an `@tool` so an agent can call it.
4. **4.4** Build the `QSARAgent` class.
5. **4.5** Talk to the agent in plain English.
6. **4.6** Optional exercise.

## 4.1 — The QSAR pipeline (plain Python)

We'll define seven helper functions. Each one does **one job**, has a clear docstring, and is small enough to read at a glance. Later we'll expose these to the agent.

First, a configuration block.

In [28]:
# Where the dataset lives. Change this if you move the file.
DATA_PATH = (
    "/Users/ricat/Documents/GitHub/CE_Course_STEMPeers/"
    "Data/STopTox Datasets/Skin_sensitization.sdf"
)

# Column names inside the SDF.
SMILES_COL = "Smiles"
LABEL_COL = "Outcome"

# Reproducibility — the same seed everywhere keeps demos predictable.
RANDOM_STATE = 42

### Helper 1 — Load the dataset

Reads the SDF, drops rows with missing labels, returns a clean `DataFrame`.

In [29]:
def load_skin_sensitization_dataset() -> pd.DataFrame:
    """Load the skin-sensitization SDF and return a clean DataFrame.

    Returns a DataFrame with two useful columns: 'Smiles' and 'Outcome'
    (Outcome is int 0 or 1).
    """
    df = PandasTools.LoadSDF(DATA_PATH, smilesName=SMILES_COL, includeFingerprints=False)
    # Drop rows where the label is missing or the SMILES is empty.
    df = df.dropna(subset=[LABEL_COL, SMILES_COL]).copy()
    df = df[df[LABEL_COL].astype(str).str.strip() != ""].copy()
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    return df.reset_index(drop=True)

### Helper 2 — SMILES → Morgan fingerprints (ECFP4)

We turn each molecule into a 2048-bit binary vector (Morgan radius 2, equivalent to **ECFP4**). Invalid SMILES are skipped.

In [30]:
def smiles_to_morgan_fingerprints(
    smiles_list,
    radius: int = 2,
    nbits: int = 2048,
):
    """Compute ECFP4-style Morgan fingerprints for a list of SMILES.

    Returns (X, valid_indices):
        X            — numpy array of shape (n_valid, nbits), dtype uint8
        valid_indices — indices into the original list that parsed successfully
    """
    fps, valid_idx = [], []
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        bv = AllChem.GetMorganFingerprintAsBitVect(
            mol, radius, nBits=nbits, useFeatures=False
        )
        arr = np.zeros((nbits,), dtype=np.uint8)
        DataStructs.ConvertToNumpyArray(bv, arr)
        fps.append(arr)
        valid_idx.append(i)
    return np.array(fps), np.array(valid_idx)

### Helper 3 — Train/test split

A standard stratified split so both classes are represented in train and test.

In [31]:
def split_train_test(X, y, test_size: float = 0.2):
    """Stratified train/test split. Returns X_train, X_test, y_train, y_test."""
    return train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=RANDOM_STATE,
    )

### Helper 4 — Train a Random Forest with Optuna

**Optuna** is a hyperparameter-search library. We let it explore a few settings (`n_estimators`, `max_depth`, etc.) and pick the best, scored by **G-Mean** (geometric mean of sensitivity and specificity — a good metric for slightly imbalanced data).

We use only **5 trials** for speed during the live workshop. In a production setting you'd use 50–100.

In [32]:
def train_random_forest(X_train, y_train, n_trials: int = 5):
    """Tune & train a Random Forest with Optuna.

    Returns a dict: {'model': fitted_estimator, 'best_params': {...},
                     'cv_score': float}.
    """
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 400),
            "max_depth":    trial.suggest_int("max_depth", 4, 30),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        }
        clf = RandomForestClassifier(**params)
        scores = []
        for train_idx, val_idx in cv.split(X_train, y_train):
            clf.fit(X_train[train_idx], y_train[train_idx])
            preds = clf.predict(X_train[val_idx])
            scores.append(geometric_mean_score(y_train[val_idx], preds))
        return float(np.mean(scores))

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best = RandomForestClassifier(**study.best_params, random_state=RANDOM_STATE, n_jobs=-1)
    best.fit(X_train, y_train)
    return {"model": best, "best_params": study.best_params, "cv_score": study.best_value}

### Helper 5 — Train an SVM with Optuna

Same recipe, but with an RBF-kernel SVM. `C` and `gamma` are searched on a log scale because that's how those hyperparameters behave.

In [33]:
def train_svm(X_train, y_train, n_trials: int = 5):
    """Tune & train an RBF-kernel SVM with Optuna.

    Returns the same dict shape as train_random_forest().
    """
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    def objective(trial):
        params = {
            "C":     trial.suggest_float("C", 1e-2, 1e2, log=True),
            "gamma": trial.suggest_float("gamma", 1e-4, 1e0, log=True),
            "kernel": "rbf",
            "probability": True,
            "random_state": RANDOM_STATE,
        }
        clf = SVC(**params)
        scores = []
        for train_idx, val_idx in cv.split(X_train, y_train):
            clf.fit(X_train[train_idx], y_train[train_idx])
            preds = clf.predict(X_train[val_idx])
            scores.append(geometric_mean_score(y_train[val_idx], preds))
        return float(np.mean(scores))

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best = SVC(**study.best_params, kernel="rbf", probability=True, random_state=RANDOM_STATE)
    best.fit(X_train, y_train)
    return {"model": best, "best_params": study.best_params, "cv_score": study.best_value}

### Helper 6 — Evaluate on holdout

Returns the same metrics Ricardo's research notebook reports, so the outputs are directly comparable to a 'real' QSAR study.

In [34]:
def evaluate_model(model, X_test, y_test) -> dict:
    """Return a dict of QSAR-style classification metrics."""
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    ppv         = tp / (tp + fp) if (tp + fp) else 0.0
    npv         = tn / (tn + fn) if (tn + fn) else 0.0
    ccr         = (sensitivity + specificity) / 2

    return {
        "bal_acc":     round(balanced_accuracy_score(y_test, pred), 3),
        "sensitivity": round(sensitivity, 3),
        "specificity": round(specificity, 3),
        "ccr":         round(ccr, 3),
        "ppv":         round(ppv, 3),
        "npv":         round(npv, 3),
        "auc":         round(roc_auc_score(y_test, proba), 3),
    }

### Helper 7 — Predict for new SMILES

Given a trained model and a list of SMILES, returns a prediction and a probability for each.

In [35]:
def predict_smiles(model, smiles_list) -> list:
    """Predict skin sensitization (1) vs non-sensitizer (0) for SMILES.

    Returns a list of dicts: {smiles, prediction, probability}.
    """
    X, valid_idx = smiles_to_morgan_fingerprints(smiles_list)
    if len(X) == 0:
        return []
    pred = model.predict(X)
    proba = model.predict_proba(X)[:, 1]
    out = []
    for i, idx in enumerate(valid_idx):
        out.append({
            "smiles": smiles_list[idx],
            "prediction": int(pred[i]),
            "probability": round(float(proba[i]), 3),
        })
    return out

## 4.2 — Run the pipeline manually first

Before we hand it to an agent, let's verify the helpers work end-to-end. **Watch your terminal/notebook output — this should take ~30–60 seconds.**

In [36]:
# 1. Load.
df = load_skin_sensitization_dataset()
print(f"Dataset: {len(df)} molecules")
print(f"Class balance:\n{df[LABEL_COL].value_counts()}")

# 2. Featurize.
X, valid_idx = smiles_to_morgan_fingerprints(df[SMILES_COL].tolist())
y = df[LABEL_COL].values[valid_idx]
print(f"\nFingerprint matrix: {X.shape}")

# 3. Split.
X_tr, X_te, y_tr, y_te = split_train_test(X, y)
print(f"Train: {X_tr.shape[0]}  |  Test: {X_te.shape[0]}")

# 4. Train a Random Forest.
print("\nTraining Random Forest with Optuna (5 trials)...")
rf_result = train_random_forest(X_tr, y_tr, n_trials=5)
print(f"Best params: {rf_result['best_params']}")
print(f"CV G-mean:   {rf_result['cv_score']:.3f}")

# 5. Evaluate.
print("\nHoldout metrics:")
print(evaluate_model(rf_result["model"], X_te, y_te))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/ricat/Documents/GitHub/CE_Course_STEMPeers/Data/STopTox Datasets/Skin_sensitization.sdf'

Good — we have a working QSAR pipeline. Now let's let an **agent** drive it.

## 4.3 — Wrap helpers as agent tools

The agent needs to call our helpers across multiple turns. We give it a small **shared workspace** — a module-level dictionary — so each tool can store its result for the next tool to pick up. Think of it as the agent's scratchpad.

In [ ]:
# Shared workspace — every QSAR tool reads/writes here.
_AGENT_STATE = {
    "df":     None,   # loaded DataFrame
    "X":      None,   # fingerprints
    "y":      None,   # labels
    "split":  None,   # (X_tr, X_te, y_tr, y_te)
    "models": {},     # {'rf': {...}, 'svm': {...}}
}

Now the tools. Each is a thin wrapper around a helper, with a docstring the LLM will use to decide *when* to call it.

In [ ]:
@tool
def load_dataset_tool() -> str:
    """Load the skin sensitization dataset from disk into the workspace.

    Call this first, before any other QSAR tool. Returns a JSON summary
    with row count and class balance.
    """
    df = load_skin_sensitization_dataset()
    _AGENT_STATE["df"] = df
    counts = df[LABEL_COL].value_counts().to_dict()
    return json.dumps({
        "success": True,
        "n_molecules": len(df),
        "class_balance": {"non_sensitizer_0": int(counts.get(0, 0)),
                            "sensitizer_1":     int(counts.get(1, 0))},
    })

In [ ]:
@tool
def featurize_tool() -> str:
    """Compute Morgan ECFP4 fingerprints for the loaded dataset.

    Requires that load_dataset_tool() was already called. Stores X and y
    in the workspace and returns the matrix shape.
    """
    if _AGENT_STATE["df"] is None:
        return json.dumps({"success": False, "error": "Call load_dataset_tool first."})
    df = _AGENT_STATE["df"]
    X, valid_idx = smiles_to_morgan_fingerprints(df[SMILES_COL].tolist())
    y = df[LABEL_COL].values[valid_idx]
    _AGENT_STATE["X"], _AGENT_STATE["y"] = X, y
    return json.dumps({
        "success": True,
        "X_shape": list(X.shape),
        "y_shape": list(y.shape),
        "n_invalid_smiles": int(len(df) - len(valid_idx)),
    })

In [ ]:
@tool
def split_data_tool(test_size: float = 0.2) -> str:
    """Stratified train/test split on the featurized data.

    Args:
        test_size: fraction held out for testing (default 0.2).

    Requires featurize_tool() was already called.
    """
    if _AGENT_STATE["X"] is None:
        return json.dumps({"success": False, "error": "Call featurize_tool first."})
    X_tr, X_te, y_tr, y_te = split_train_test(_AGENT_STATE["X"], _AGENT_STATE["y"], test_size)
    _AGENT_STATE["split"] = (X_tr, X_te, y_tr, y_te)
    return json.dumps({
        "success": True,
        "n_train": int(X_tr.shape[0]),
        "n_test":  int(X_te.shape[0]),
    })

In [ ]:
@tool
def train_and_evaluate_tool(model_type: str, n_trials: int = 5) -> str:
    """Train a QSAR classifier with Optuna and evaluate on the holdout set.

    Args:
        model_type: 'rf' for Random Forest, or 'svm' for RBF-kernel SVM.
        n_trials:   number of Optuna trials (default 5; keep <=10 for live demos).

    Requires split_data_tool() was already called. Stores the trained
    model in the workspace and returns metrics + best hyperparameters.
    """
    if _AGENT_STATE["split"] is None:
        return json.dumps({"success": False, "error": "Call split_data_tool first."})
    X_tr, X_te, y_tr, y_te = _AGENT_STATE["split"]

    if model_type == "rf":
        result = train_random_forest(X_tr, y_tr, n_trials=n_trials)
    elif model_type == "svm":
        result = train_svm(X_tr, y_tr, n_trials=n_trials)
    else:
        return json.dumps({"success": False, "error": f"Unknown model_type '{model_type}'. Use 'rf' or 'svm'."})

    metrics = evaluate_model(result["model"], X_te, y_te)
    _AGENT_STATE["models"][model_type] = result
    return json.dumps({
        "success": True,
        "model_type": model_type,
        "best_params": result["best_params"],
        "cv_gmean":    round(result["cv_score"], 3),
        "holdout_metrics": metrics,
    })

In [ ]:
@tool
def benchmark_models_tool(n_trials: int = 5) -> str:
    """Benchmark Random Forest vs SVM side-by-side.

    Trains both models with Optuna and returns their holdout metrics in
    one comparison object. Requires split_data_tool() was already called.

    Args:
        n_trials: Optuna trials per model (default 5).
    """
    if _AGENT_STATE["split"] is None:
        return json.dumps({"success": False, "error": "Call split_data_tool first."})

    rf_resp = json.loads(train_and_evaluate_tool.invoke({"model_type": "rf",  "n_trials": n_trials}))
    svm_resp = json.loads(train_and_evaluate_tool.invoke({"model_type": "svm", "n_trials": n_trials}))
    return json.dumps({
        "success": True,
        "random_forest": rf_resp,
        "svm":           svm_resp,
    }, indent=2)

In [ ]:
@tool
def predict_smiles_tool(smiles: str, model_type: str = "rf") -> str:
    """Predict skin sensitization for one SMILES using a previously trained model.

    Args:
        smiles:     a single SMILES string.
        model_type: which trained model to use ('rf' or 'svm').

    Returns prediction (0 or 1) and probability of being a sensitizer.
    """
    if model_type not in _AGENT_STATE["models"]:
        return json.dumps({
            "success": False,
            "error": f"No '{model_type}' model trained yet. Train one first with train_and_evaluate_tool.",
        })
    model = _AGENT_STATE["models"][model_type]["model"]
    preds = predict_smiles(model, [smiles])
    if not preds:
        return json.dumps({"success": False, "error": "Invalid SMILES."})
    p = preds[0]
    p["label"] = "sensitizer" if p["prediction"] == 1 else "non-sensitizer"
    p["success"] = True
    return json.dumps(p)

## 4.4 — The QSAR Agent

Same LangGraph pattern as before. The two new pieces:

- A **system prompt** that tells the agent who it is and which tools it has.
- **Memory** (`MemorySaver`) so the agent remembers prior turns within a session.

In [ ]:
QSAR_SYSTEM_PROMPT = """You are QSAR Modeler, a specialized AI assistant for
building and evaluating QSAR machine-learning models for skin sensitization.

You have access to tools that:
  - Load the skin-sensitization dataset.
  - Featurize molecules using Morgan ECFP4 fingerprints.
  - Split data into train and test.
  - Train and evaluate Random Forest or SVM classifiers (with Optuna).
  - Benchmark RF vs SVM side-by-side.
  - Predict skin sensitization for a new SMILES string.

Operating rules:
  1. Always call tools in a sensible order: load → featurize → split →
     train → evaluate → predict. Skip steps only if the user explicitly
     says they have already been done in this session.
  2. When reporting metrics, present them clearly and explain what each
     metric (sensitivity, specificity, balanced accuracy, AUC) means in
     plain language.
  3. If you compare models, give a one-sentence verdict on which is better
     and *why* (looking at the holdout metrics, not just CV).
  4. If a SMILES is predicted to be a sensitizer with high probability,
     flag it explicitly and remind the user this is a model prediction,
     not an experimental result.
"""

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[List[AnyMessage], add_messages]


class QSARAgent:
    """A LangGraph agent that builds and evaluates QSAR models."""

    def __init__(self):
        self.system_prompt = QSAR_SYSTEM_PROMPT
        self.tools = [
            load_dataset_tool,
            featurize_tool,
            split_data_tool,
            train_and_evaluate_tool,
            benchmark_models_tool,
            predict_smiles_tool,
        ]
        self.llm = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash",
            temperature=0,
            google_api_key=os.environ.get("GOOGLE_API_KEY"),
            convert_system_message_to_human=True,
        )
        self.llm_with_tools = self.llm.bind_tools(self.tools)

        graph = StateGraph(AgentState)
        graph.add_node("assistant", self._call_model)
        graph.add_node("tools", ToolNode(self.tools))
        graph.add_conditional_edges("assistant", tools_condition)
        graph.add_edge("tools", "assistant")
        graph.set_entry_point("assistant")

        # MemorySaver lets the agent remember turn-to-turn within a thread.
        self.graph = graph.compile(checkpointer=MemorySaver())

    def _call_model(self, state: AgentState):
        messages = [SystemMessage(content=self.system_prompt)] + state["messages"]
        return {"messages": [self.llm_with_tools.invoke(messages)]}

    def run(self, query: str, thread_id: str = "default") -> str:
        config = {"configurable": {"thread_id": thread_id}}
        result = self.graph.invoke(
            {"messages": [HumanMessage(content=query)]},
            config=config,
        )
        return result["messages"][-1].content

In [ ]:
qsar_agent = QSARAgent()
print("✅ QSAR agent ready.")

## 4.5 — Talk to the agent

Four demos. After each one, look at:
1. **Did the agent call the right tools?**
2. **Did it explain the result clearly?**
3. **Did it remember earlier steps?** (because of `MemorySaver`)

### Demo 1 — Get to know the data

In [ ]:
response = qsar_agent.run(
    "Load the skin sensitization dataset and tell me about the class balance."
)
print(response)

### Demo 2 — Build a Random Forest

In [ ]:
response = qsar_agent.run(
    "Now featurize the molecules with Morgan fingerprints, split the data "
    "into train and test, then train a Random Forest model and report all the metrics."
)
print(response)

### Demo 3 — Compare RF vs SVM

Notice the agent reuses the already-loaded data because of memory.

In [ ]:
response = qsar_agent.run(
    "Now also train an SVM and compare it side-by-side with the Random Forest. "
    "Which performs better, and why might that be?"
)
print(response)

### Demo 4 — Predict a known sensitizer

**2,4-dinitrochlorobenzene (DNCB)** is a textbook strong skin sensitizer (used as a positive control in immunology research). Let's see if our model agrees.

In [ ]:
response = qsar_agent.run(
    "Predict whether 2,4-dinitrochlorobenzene "
    "(SMILES: O=[N+]([O-])c1ccc(Cl)c([N+](=O)[O-])c1) is a skin sensitizer "
    "using the Random Forest model."
)
print(response)

## 4.6 — Optional Exercise: screen a custom SMILES list

Build a new tool that accepts a comma-separated list of SMILES, predicts skin-sensitization probability for each using the trained model, and returns them ranked from most to least likely sensitizer.

**Hints:**
- Reuse the `predict_smiles` helper.
- Sort the resulting list by `"probability"` in descending order.
- Return JSON so the agent can quote it cleanly.

Once you've implemented the tool, register it with the agent like this:
```python
qsar_agent.tools.append(screen_smiles_list_tool)
qsar_agent.llm_with_tools = qsar_agent.llm.bind_tools(qsar_agent.tools)
```
Then ask the agent something like: *"Screen this SMILES list and tell me which compound is most likely to be a skin sensitizer."*

In [ ]:
@tool
def screen_smiles_list_tool(smiles_csv: str, model_type: str = "rf") -> str:
    """Predict skin-sensitization probability for many SMILES at once and rank them.

    Args:
        smiles_csv: comma-separated SMILES strings, e.g. "CCO,CC(=O)O,c1ccccc1".
        model_type: which trained model to use ('rf' or 'svm').

    Returns a JSON-encoded list of {smiles, prediction, probability},
    sorted by probability descending.
    """
    # ---- YOUR CODE HERE -------------------------------------------------
    # 1. Check that a model of model_type exists in _AGENT_STATE['models'].
    # 2. Split smiles_csv on ',' and strip whitespace.
    # 3. Use predict_smiles(model, smiles_list) to get predictions.
    # 4. Sort by probability, descending.
    # 5. Return json.dumps(...) of the sorted list.
    # ---------------------------------------------------------------------
    return json.dumps({"success": False, "error": "Not implemented yet — your turn!"})

## Where to go next

What we built today is a small but real agentic QSAR system. To take it closer to production you'd add:

- **Applicability domain (AD) scoring** — flag molecules that are too different from the training set to be predicted reliably.
- **Calibrated thresholds** — instead of `predict_proba >= 0.5`, tune the threshold on a ROC curve to maximize G-mean.
- **Multi-endpoint orchestration** — give the agent tools for skin-irritation, eye-irritation, and acute-toxicity datasets and let it build a full hazard profile for a new molecule.

These are great extensions for a follow-up workshop — the agent framework here generalizes to any ML pipeline you can wrap as a function.